# <center>Getting English videos</center>

*Seleciona apenas os vídeos do youtube em inglês*

---

This code separates only English videos

In [6]:
!pip install nltk langdetect

In [2]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import nltk
from langdetect import detect, DetectorFactory
from multiprocessing import Pool, cpu_count
import os
import sys

nltk.download("stopwords")
stop_words = stopwords.words("english")

vectorizer = TfidfVectorizer(
    stop_words=stop_words,
    ngram_range=(1, 2),   # unigrams + bigrams (captura combinações de palavras)
    max_features=5000     # limita vocabulário para economizar memória
)

project_root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root_path not in sys.path:
    sys.path.append(project_root_path)

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/students/moliveira/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Getting the CSV that has the informations of all videos

In [3]:
df = pd.read_csv("youtube_titulos_saida.csv", usecols=["video_id", "title", "description"])
display(df)

,video_id,title,description
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a..."
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron..."
3,TwACgO_oPq4,Anacondaz — Акуле плевать (Official Music Video),#Anacondaz #Акулеплевать #Безпаники\n\nLong St...
4,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...
...,...,...,...
1183811,J2FGKFv_xbs,Ministra de Relações Exteriores da África do S...,"""Os agentes israelenses tentam intimidar, mas ..."
1183812,fNTfEPLoAik,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...
1183813,8pEHwRiJVJM,What’s your thougts? #Bible #god #Christianscr...,What’s your thoughts?\n\nFULL VIDEO: 👇🏽\nhttps...
1183814,L_y36luTTok,Did you know this ? #gaza #god #israel #jesus ...,Have you seen this ? \n\nFULL VIDEO: Aries int...


In [4]:
valores_nulos = df.isnull().sum()
print(valores_nulos)

video_id            0
title               1
description    107752
dtype: int64


Leaving only English titles

In [5]:
DetectorFactory.seed = 0

def detect_language(text):
    if not isinstance(text, str) or text.strip() == "":
        return "unknown"
    try:
        return detect(text)
    except:
        return "unknown"
    
def parallel_detect(texts, num_workers=None):
    if num_workers is None:
        num_workers = cpu_count()  # usa todos os núcleos disponíveis
    with Pool(num_workers) as pool:
        results = pool.map(detect_language, texts)
    return results


df["title_description"] = (df["title"].fillna("") + " " + df["description"].fillna("")).str.strip()
df["lang"] = parallel_detect(df["title_description"].tolist())

# Descomentar essas linhas se for realizar o processo de novo
#df["lang"] = parallel_detect(df["title"].tolist())
df_en = df[df["lang"] == "en"]
df_en.to_csv("data/english_videos_infos.csv", index=False)

df_en = pd.read_csv("data/english_videos_infos.csv")

print("Total de vídeos coletados:", len(df))
print("Total de vídeos em inglês:", len(df_en))
display(df)
display(df_en)


Total de vídeos coletados: 1183816
Total de vídeos em inglês: 686626


,video_id,title,description,title_description,lang
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Free bus travel for migrants scrapped. For 5 m...,en
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",What is Spiritual Warfare? This charge I commi...,en
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...","80 Putins Have Layers In February 2024, Tucker...",en
3,TwACgO_oPq4,Anacondaz — Акуле плевать (Official Music Video),#Anacondaz #Акулеплевать #Безпаники\n\nLong St...,Anacondaz — Акуле плевать (Official Music Vide...,ru
4,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,en
...,...,...,...,...,...
1183811,J2FGKFv_xbs,Ministra de Relações Exteriores da África do S...,"""Os agentes israelenses tentam intimidar, mas ...",Ministra de Relações Exteriores da África do S...,pt
1183812,fNTfEPLoAik,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,en
1183813,8pEHwRiJVJM,What’s your thougts? #Bible #god #Christianscr...,What’s your thoughts?\n\nFULL VIDEO: 👇🏽\nhttps...,What’s your thougts? #Bible #god #Christianscr...,en
1183814,L_y36luTTok,Did you know this ? #gaza #god #israel #jesus ...,Have you seen this ? \n\nFULL VIDEO: Aries int...,Did you know this ? #gaza #god #israel #jesus ...,en


,video_id,title,description,title_description,lang
0,jEKzQV5oajY,Free bus travel for migrants scrapped. For 5 m...,NaN,Free bus travel for migrants scrapped. For 5 m...,en
1,xrGGce8cmx8,What is Spiritual Warfare?,"This charge I commit unto thee, son Timothy, a...",What is Spiritual Warfare? This charge I commi...,en
2,uaozGpSc4nc,80 Putins Have Layers,"In February 2024, Tucker Carlson stood in fron...","80 Putins Have Layers In February 2024, Tucker...",en
3,A59ftbhsQUE,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,#pleiadians #5dimension #ashtar\n#pleiadians #...,𝐆𝐀𝐋𝐀𝐂𝐓𝐈𝐂 𝐀𝐋𝐋𝐈𝐀𝐍𝐂𝐄 𝐌𝐄𝐒𝐒𝐀𝐆𝐄|𝐁𝐈𝐆 𝐍𝐄𝐖𝐒!~𝐑𝐄𝐌𝐀𝐈𝐍 𝐀𝐋𝐄...,en
4,AKU0RokegSo,Los Angeles rain: Studio City homes evacuated ...,An atmospheric river storm has caused mudslide...,Los Angeles rain: Studio City homes evacuated ...,en
...,...,...,...,...,...
686621,xV2G3GFf2ug,Did you know this ? #gaza #god #israel #jesus ...,Have you seen this ? \n\nFULL VIDEO: Aries int...,Did you know this ? #gaza #god #israel #jesus ...,en
686622,GEhEUy85PsY,Meet the balkans,This is a video,Meet the balkans This is a video,en
686623,fNTfEPLoAik,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,Former Clark County sheriff arrested on 15 fel...,en
686624,8pEHwRiJVJM,What’s your thougts? #Bible #god #Christianscr...,What’s your thoughts?\n\nFULL VIDEO: 👇🏽\nhttps...,What’s your thougts? #Bible #god #Christianscr...,en


In [ ]:
df_total = pd.read_csv("data/